In [ ]:
# ===== Map fitting: T1 / T2 / PD from the 96 TI frames (Ehses et al. 2013 model) =====
#
# Model (Ehses et al., Magn. Reson. Med. 69:71, 2013, Eq. 1) -- signed mono-exponential:
#
#     S(TI) = S_St - (S_0 + S_St) * exp(-TI / T1*)
#
#     S_0  : transient amplitude extrapolated to TI=0  (signal just after inversion,
#            ~ M0 * flip-dependent factor)
#     S_St : bSSFP steady-state amplitude
#     T1*  : apparent (MT-/bSSFP-affected) longitudinal relaxation time
#
# T1, T2, M0 are then derived from the 3 fit params + the known flip angle (Eqs. 2-4):
#
#     T1  = T1* * (S_0 / S_St) * cos(alpha/2)
#     T2  = T1* * sin^2(alpha/2) / (1 - S_St / (S_0 * cos(alpha/2)))
#     M0  = S_0 * sin(alpha/2)
#
# WHY 3 PARAMETERS (this is what the earlier 2-param model got wrong):
#   The old model S(TI) = D*(1 - 2e^{-TI/T1}) has ONE free complex scale, forcing
#   |S(0)| = |S(inf)|. That locks away the S_0/S_St ratio -- and that ratio is
#   ~ 1/G(T1,T2), the only place T2 lives at a single flip angle. With S_0 and S_St
#   fit INDEPENDENTLY, T2 becomes identifiable from the same single-flip-angle data.
#
# FIT: grid over T1* (150 log-spaced values); for each, (S_0, S_St) are solved by a
#      CLOSED-FORM 2x2 complex least squares per voxel (the model is complex-linear in
#      P = S_St*e^{ipsi} and Q = S_0*e^{ipsi}, so the unknown image phase psi is
#      absorbed automatically -- no phase alignment needed, robust to coil-combine
#      phase and to the IR null sign flip). Fully vectorized over all voxels.
#
# KNOWN LIMITS (inherited from the paper):
#   * Assumes TR >> T2 (steady-state relation). Ours: TR=7.8 ms vs T2=70/80 ms --
#     marginal for WM/GM, fine for CSF -> expect small T2 bias in white matter.
#   * In vivo, the paper corrects slice profile (effective flip angle, lookup table)
#     and MT (3 pulse-duration scans, extrapolate S_St). Our 3D sim has no slice
#     profile and no MT, so those corrections are not needed here.
#   * M0 is only accurate up to the (arbitrary) global recon scale, same as PD before.
#   * The first 1-2 frames use an asymmetric KWIC window (no data before t=0) and are
#     the least reliable; the fit uses all frames (robust 96-point LSQ).
%matplotlib widget
import sys, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

def _find_repo_root(start):
    cur = start.resolve()
    for d in [cur, *cur.parents]:
        if (d / "Modules").is_dir():
            return d
    raise RuntimeError(f"Could not find a folder containing 'Modules' from {cur}")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Modules import PARAMS, build_sim_data

p = PARAMS
n = p.n
frame_advance = 4                       # MUST match the training/KWIC value
centers = np.arange(0, p.N_readout, frame_advance)
ti_frames = p.TI_START + centers * p.P_echo      # (96,) seconds
F_ALL = len(centers)
TR = p.TR_echo                             # bSSFP steady-state TR
ALPHA = p.alpha_deg
print(f"n={n}  F_ALL={F_ALL}  TI {ti_frames[0]*1e3:.0f}..{ti_frames[-1]*1e3:.0f} ms  "
      f"TR={TR*1e3:.1f} ms  alpha={ALPHA:.0f} deg")
print(f"IR nulls (T1*ln2): WM {0.9*np.log(2)*1e3:.0f} ms, GM {1.3*np.log(2)*1e3:.0f} ms, "
      f"CSF {4.0*np.log(2)*1e3:.0f} ms  (all inside the TI window -> sign flip present)")
print(f"TR/T2 check (Ehses assumes TR>>T2):  WM {TR/0.07:.2f}  GM {TR/0.08:.2f}  CSF {TR/0.5:.2f}")

In [ ]:
# ---- load reconstruction (final sweep) + ground-truth masks ----
frames_path = REPO_ROOT / "Training & Inference" / "Trained models" / "zs3d300_frames.pt"
zs3d = torch.load(frames_path, map_location="cpu")          # {f: (n,n,n) complex}
S = torch.stack([zs3d[f] for f in range(F_ALL)], dim=-1).to(torch.complex128)  # (n,n,n,F)
S_flat = S.contiguous().view(-1, F_ALL)                      # (V, F) complex
print("S:", tuple(S.shape), "  V =", S_flat.shape[0])

data, masks, brain_mask, coil_den, coil_c64 = build_sim_data(
    raw_path=str(REPO_ROOT / "Data" / "subject04_crisp_v.rawb"),
    p=PARAMS, device=torch.device("cpu"), permute_order=(2, 1, 0))
print("masks:", {k: int(v.sum()) for k, v in masks.items()})

In [ ]:
# ---- Ehses model pieces (vectorized over all voxels) ----
def ehses_derive(S0, Sst, T1s, alpha=60.0):
    """Ehses Eqs. 2-4: map the 3 fit params to (T1, T2, M0). All args arrays/scalars."""
    ar = np.deg2rad(alpha)
    T1  = T1s * (S0 / Sst) * np.cos(ar/2)
    den = 1.0 - Sst / (S0 * np.cos(ar/2))
    T2  = np.where(np.abs(den) > 1e-9, T1s * np.sin(ar/2)**2 / den, np.nan)
    M0  = S0 * np.sin(ar/2)
    return T1, T2, M0

def fit_ehses(S, ti, T1_grid):
    """Closed-form phase-robust fit of S(TI) = Sst - (S0+Sst) e^{-TI/T1*} per voxel.

    For each T1* on the grid the model is complex-linear in
        P = Sst * e^{ipsi},  Q = S0 * e^{ipsi}
    so the per-voxel optimum is a 2x2 complex normal-equation solve, and the unknown
    per-voxel image phase psi is absorbed automatically (works through the IR null).

    S : (V, F) complex   ti : (F,) seconds
    Returns T1s, S0, Sst, resid2 : (V,) arrays.
    """
    V, F = S.shape
    s2 = (np.abs(S) ** 2).sum(1).real
    best = np.full(V, np.inf); bT1 = np.zeros(V)
    bS0 = np.zeros(V); bSst = np.zeros(V)
    for T1s in T1_grid:
        e = np.exp(-ti / T1s)
        A0 = (1.0 - e).astype(np.complex128)     # Sst coefficient
        A1 = (-e).astype(np.complex128)          # S0  coefficient
        n00 = (np.abs(A0) ** 2).sum();  n11 = (np.abs(A1) ** 2).sum()
        n01 = (A0 * np.conj(A1)).sum()
        GS0 = (S * A0).sum(1);  GS1 = (S * A1).sum(1)      # (V,) each
        GA = np.array([[n00, n01], [np.conj(n01), n11]])
        X = np.linalg.solve(GA, np.stack([GS0, GS1], axis=1).T).T   # (V,2) = [P, Q]
        pred2 = (np.abs(X[:,0])**2 * n00 + np.abs(X[:,1])**2 * n11
                 + 2 * np.real(X[:,0] * np.conj(X[:,1]) * n01))
        res2 = s2 - pred2
        m = res2 < best
        if m.any():
            best[m] = res2[m];  bT1[m] = T1s
            bSst[m] = np.abs(X[m,0]);  bS0[m] = np.abs(X[m,1])
    return bT1, bS0, bSst, best

In [ ]:
t0 = time.time()
T1_grid = np.logspace(np.log10(0.15), np.log10(8.0), 150)
T1s, S0, Sst, resid2 = fit_ehses(S_flat.numpy(), ti_frames, T1_grid)
T1, T2, M0 = ehses_derive(S0, Sst, T1s, ALPHA)
T1map, T2map, PDmap = T1.reshape(n,n,n), T2.reshape(n,n,n), M0.reshape(n,n,n)
s2 = (np.abs(S_flat.numpy() ** 2)).sum(axis=1)
print(f"Ehses fit done in {time.time()-t0:.0f} s   (mean rel-resid {np.sqrt(resid2/s2).mean():.3f})")
print(f"T1* range: {T1s.min():.3f} .. {T1s.max():.3f} s")

In [ ]:
# ---- validate against ground-truth tissues ----
gt = {"WM": (0.9, 0.07, 0.7), "GM": (1.3, 0.08, 0.8), "CSF": (4.0, 0.5, 1.0)}
print(f"{'tissue':6} {'T1 fit':>8} {'T1 gt':>6} {'err%':>6} | {'T2 fit':>8} {'T2 gt':>6} {'err%':>6} | {'PD fit':>8} {'PD gt':>6}")
for t in ("WM", "GM", "CSF"):
    m = masks[t]
    t1t, t2t, pdt = gt[t]
    t1f, t2f, pdf = T1map[m].mean(), T2map[m].mean(), PDmap[m].mean()
    print(f"{t:6} {t1f:8.3f} {t1t:6.2f} {100*(t1f-t1t)/t1t:+6.1f} | "
          f"{t2f:8.3f} {t2t:6.2f} {100*(t2f-t2t)/t2t:+6.1f} | {pdf:8.3f} {pdt:6.2f}")
print()
print("Caveats: T2 from Ehses Eq. 3 assumes TR>>T2 (marginal at our TR=7.8ms for WM/GM).")
print("PD = M0 = S0*sin(alpha/2): accurate only up to the global recon scale.")

In [ ]:
# ---- display T1 / T2 / PD mid-slices ----
mid = n // 2
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, m, title, cmap, lo, hi in [
    (ax[0], T1map, "T1 (s)",      "inferno", 0.2, 6.0),
    (ax[1], T2map, "T2 (s)",      "inferno", 0.02, 1.0),
    (ax[2], PDmap, "PD (rel.)",   "gray",    0.0, None)]:
    sl = np.rot90(m[:, :, mid].T, 2)
    v = hi if hi is not None else np.percentile(m[brain_mask], 99)
    a.imshow(sl, cmap=cmap, vmin=lo, vmax=v, extent=[-125, 125, -125, 125])
    a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()